In [12]:
import pandas as pd
import numpy as np

AREA         = 40_000
GAMMA        = -0.004
T_REF        = 25
INTERVAL_HRS = 0.25

# ⚠️ ROOT CAUSE FIX:
# 'Fifteen_minutes_merged_MEMR_data.xlsx' only covers 2020–2021.
# The LSTM prediction files cover 2022 only.
# → Use 'Fifteen_minutes_merged_2022_data.xlsx' which covers Jan–Feb 2022.
measured = pd.read_excel('/content/Fifteen_minutes_merged_2022_data.xlsx', index_col='datetime', parse_dates=True)
irr_pred = pd.read_csv('/content/lstm_predictions_irradiance_2022.csv', index_col='datetime', parse_dates=True)
tmp_pred = pd.read_csv('/content/lstm_temperature_predictions_2022_data_droup02.csv', index_col='datetime', parse_dates=True)

# Sanity check before merging — prints 0 if there's a date mismatch
print("measured  :", measured.index.min(), "→", measured.index.max())
print("irr_pred  :", irr_pred.index.min(), "→", irr_pred.index.max())
print("tmp_pred  :", tmp_pred.index.min(), "→", tmp_pred.index.max())
print("Overlap check (should be > 0):", len(measured.index.intersection(irr_pred.index)))

df = measured.join(irr_pred, how='inner').join(tmp_pred, how='inner')
print(f"Rows after merge: {len(df):,}")

# Step 1 — Energy → Power
df['power_kw'] = df['generated_yield'] / INTERVAL_HRS

# Step 2 — Fix irradiance units
df['irradiance_kw_m2'] = df['irradiance_avg'] / 1000

# Step 3–4 — Efficiency
df['efficiency'] = df['power_kw'] / (df['irradiance_kw_m2'] * AREA)
df.loc[df['irradiance_avg'] < 50, 'efficiency'] = pd.NA
df['efficiency'] = df['efficiency'].clip(0, 1)

# Step 5 — η_ref
eta_ref = df[df['irradiance_avg'] > 200]['efficiency'].mean()
print(f"η_ref = {eta_ref:.6f}")

# Step 6 — Errors
df['irradiance_error']     = df['actual_irradiance']  - df['predicted_irradiance']
df['irradiance_abs_error'] = df['irradiance_error'].abs()
df['temp_error']           = df['actual_temperature'] - df['predicted_temperature']
df['temp_abs_error']       = df['temp_error'].abs()

# Step 7 — Physical PV model: P = G × A × η_ref × (1 + γ(T − 25))
df['predicted_irradiance_kw_m2'] = df['predicted_irradiance'] / 1000
df['Predicted_Energy_Yield'] = (
    df['predicted_irradiance_kw_m2']
    * AREA
    * eta_ref
    * (1 + GAMMA * (df['predicted_temperature'] - T_REF))
)

# Step 8 — Actual Energy Yield using actual irradiance and temperature
df['actual_irradiance_kw_m2'] = df['actual_irradiance'] / 1000
df['Actual_Energy_Yield'] = (
    df['actual_irradiance_kw_m2']
    * AREA
    * eta_ref
    * (1 + GAMMA * (df['actual_temperature'] - T_REF))
)

# Step 9 — Error between Actual and Predicted Energy Yield
df['energy_yield_error']     = df['Actual_Energy_Yield'] - df['Predicted_Energy_Yield']
df['energy_yield_abs_error'] = df['energy_yield_error'].abs()

df.reset_index(inplace=True)
df.to_csv('combined_energy_dataset_2022_full.csv', index=False)
print("✅ Done")
df.head()

measured  : 2022-01-01 00:00:00 → 2022-02-24 23:45:00
irr_pred  : 2022-01-01 06:00:00 → 2022-02-24 23:45:00
tmp_pred  : 2022-01-07 12:00:00 → 2022-02-24 23:45:00
Overlap check (should be > 0): 5256
Rows after merge: 4,656
η_ref = 0.486558
✅ Done


,datetime,irr_sensor_1,irr_sensor_2,irr_sensor_3,irr_sensor_4,irr_sensor_5,irr_sensor_6,irradiation_tilted,power_analyzer,generated_yield,...,irradiance_error,irradiance_abs_error,temp_error,temp_abs_error,predicted_irradiance_kw_m2,Predicted_Energy_Yield,actual_irradiance_kw_m2,Actual_Energy_Yield,energy_yield_error,energy_yield_abs_error
0,2022-01-07 12:00:00,836.280000,858.406667,794.560000,773.333333,784.893333,830.893333,813.061111,-4305.776667,4305.776667,...,54.24020,54.24020,1.239017,1.239017,0.758821,14114.724295,0.813061,15045.213814,930.489519,930.489519
1,2022-01-07 12:15:00,826.420000,846.253333,782.680000,760.593333,770.606667,817.186667,800.623333,-4227.555467,4227.555467,...,68.05920,68.05920,2.139952,2.139952,0.732564,13608.814273,0.800623,14739.768469,1130.954196,1130.954196
2,2022-01-07 12:30:00,812.660000,829.960000,766.733333,743.446667,751.946667,798.846667,783.932222,-4143.750000,4143.750000,...,72.75531,72.75531,2.393731,2.393731,0.711177,13207.469587,0.783932,14412.543225,1205.073638,1205.073638
3,2022-01-07 12:45:00,793.066667,808.313333,745.440000,721.173333,727.993333,775.466667,761.908889,-4033.643267,4033.643267,...,82.21920,82.21920,2.362067,2.362067,0.679690,12592.815147,0.761909,13976.011214,1383.196066,1383.196066
4,2022-01-07 13:00:00,769.046667,782.006667,719.820000,694.566667,700.066667,747.793333,735.550000,-3910.462200,3910.462200,...,85.39820,85.39820,1.852088,1.852088,0.650152,12022.917427,0.735550,13496.087390,1473.169963,1473.169963
